In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

In [18]:
df = pd.read_csv("../data/os_scheduling_dataset.csv")

X = df.drop(columns=["processes", "best_algorithm","long_job_ratio","arrival_range"],axis=1)
y = df["best_algorithm"]

# safety
X["burst_time_skewness"] = X["burst_time_skewness"].fillna(0)

In [19]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Mapping (IMPORTANT for understanding)
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Label Mapping:", label_mapping)

Label Mapping: {'FCFS': 0, 'Priority': 1, 'Round Robin': 2, 'SJF': 3, 'SRTF': 4}


In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [21]:
param_grid = {
    "n_estimators": [100, 150],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.08, 0.1],
    "subsample": [0.7, 0.8],
    "colsample_bytree": [0.7, 0.8],
}

In [22]:
xgb = XGBClassifier(
    eval_metric="mlogloss",
    use_label_encoder=False,
    random_state=42
)

In [23]:
grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="accuracy",
    cv=3,                 # 3-fold CV
    verbose=2,
    n_jobs=-1
)

grid.fit(X_train, y_train)

Fitting 3 folds for each of 72 candidates, totalling 216 fits


c:\Users\SOHAM\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [20:05:39] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


GridSearchCV(cv=3,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False,
                                     eval_metric='mlogloss', feature_types=None,
                                     feature_weights=None, gamma=None,
                                     grow_policy=None, importance_type=None,
                                     interaction_constrain...
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.7, 0.8],
                         'learning_rate': [0.05, 0.08, 0.1],
                         'max_depth': [4, 6, 8], 'n_estimators': [100, 150],
                         'subsample': [0.7, 0.8]},
             scoring='accuracy', verbose=2)

In [24]:
print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

Best Parameters: {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 150, 'subsample': 0.8}
Best CV Score: 0.6708333333333333


In [25]:
best_model = grid.best_estimator_

In [26]:
y_pred = best_model.predict(X_test)

# Decode labels
y_test_labels = le.inverse_transform(y_test)
y_pred_labels = le.inverse_transform(y_pred)

print("Test Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test_labels, y_pred_labels))

Test Accuracy: 0.6695833333333333

Classification Report:
              precision    recall  f1-score   support

        FCFS       0.55      0.60      0.57       614
    Priority       1.00      1.00      1.00       360
 Round Robin       0.60      0.57      0.58       626
         SJF       0.46      0.44      0.45       493
        SRTF       1.00      1.00      1.00       307

    accuracy                           0.67      2400
   macro avg       0.72      0.72      0.72      2400
weighted avg       0.67      0.67      0.67      2400

